# 09 · Doubly Linked List

In a **doubly linked list (DLL)** every node has **two** pointers: `next` (forward) and `prev` (backward).

```
None ← [1] ⇄ [2] ⇄ [3] → None
       head
```

| | Singly | Doubly |
|---|---|---|
| Move backward | ❌ | ✅ |
| Delete a node when you only have that node | needs the previous node | uses `node.prev` directly |
| Memory per node | 1 pointer | 2 pointers |

**Golden rule:** whenever you change a `next` link, update the matching `prev` link too.

**Topics covered**
1. `DoublyLinkedList` class (insert / delete / traverse)
2. Reverse a DLL
3. Remove all occurrences of a key
4. Find pairs with a given sum (brute / hash / two pointers)
5. Remove duplicates from a sorted DLL

## 1. The `DoublyLinkedList` class

| Method | What it does |
|---|---|
| `insert_at_head(val)` | new node becomes head; old head's `prev` points to it |
| `append(val)` | walk to the tail, link both directions |
| `insert_at(val, pos)` | stop at the node before `pos`, then fix **four** pointers |
| `delete_head()` / `delete_last()` | unlink at either end |
| `delete_in_between(pos)` | `node.prev.next = node.next` and `node.next.prev = node.prev` |
| `traverse()` | print `a <-> b <-> c` |

Insert `new` after `current` (and before `current.next`):
```
new.next = current.next
new.prev = current
if current.next: current.next.prev = new
current.next = new
```

In [1]:
class Node:
    def __init__(self, val):
        self.val = val
        self.next = None
        self.prev = None

class DoublyLinkedList:
    def __init__(self):
        self.head = None

    def insert_at_head(self, val):
        new_node = Node(val)
        if not self.head:
            self.head = new_node
        else:
            new_node.next = self.head
            self.head.prev = new_node
            self.head = new_node

    def append(self, val):
        new_node = Node(val)
        if not self.head:
            self.head = new_node
            return
        current = self.head
        while current.next:
            current = current.next
        current.next = new_node
        new_node.prev = current

    def insert_at(self, val, position):
        if position == 0:
            self.insert_at_head(val)
            return
        current = self.head
        count = 0
        while current and count < position - 1:     # stop at the node BEFORE the position
            current = current.next
            count += 1
        if current is None:
            print("Position out of bounds")
            return
        new_node = Node(val)
        new_node.next = current.next
        new_node.prev = current
        if current.next:                            # not inserting at the very end
            current.next.prev = new_node
        current.next = new_node

    def traverse(self):
        current = self.head
        elements = []
        while current:
            elements.append(str(current.val))
            current = current.next
        print(" <-> ".join(elements) if elements else "Empty List")

    def delete_head(self):
        if not self.head:
            print("List is empty")
            return
        if not self.head.next:                      # only one node
            self.head = None
        else:
            self.head = self.head.next
            self.head.prev = None

    def delete_last(self):
        if not self.head:
            print("List is empty")
            return
        if not self.head.next:
            self.head = None
            return
        current = self.head
        while current.next:
            current = current.next
        current.prev.next = None                    # the DLL advantage: we can step back via prev

    def delete_in_between(self, position):
        if not self.head:
            print("List is empty")
            return
        if position == 0:
            self.delete_head()
            return
        current = self.head
        count = 0
        while current and count < position:
            current = current.next
            count += 1
        if current is None:
            print("Position out of bounds")
            return
        if current.next is None:                    # it was the last node
            self.delete_last()
        else:
            current.prev.next = current.next        # bypass the node in both directions
            current.next.prev = current.prev

In [2]:
dll = DoublyLinkedList()

print("--- 1. Append (223, 10, 20) ---")
dll.append(223); dll.append(10); dll.append(20)
dll.traverse()

print("\n--- 2. Insert at head (5) ---")
dll.insert_at_head(5)
dll.traverse()

print("\n--- 3. Insert 15 at position 2 ---")
dll.insert_at(15, 2)
dll.traverse()

print("\n--- 4. Delete head ---")
dll.delete_head()
dll.traverse()

print("\n--- 5. Delete last ---")
dll.delete_last()
dll.traverse()

print("\n--- 6. Delete at position 1 ---")
dll.delete_in_between(1)
dll.traverse()

--- 1. Append (223, 10, 20) ---
223 <-> 10 <-> 20

--- 2. Insert at head (5) ---
5 <-> 223 <-> 10 <-> 20

--- 3. Insert 15 at position 2 ---
5 <-> 223 <-> 15 <-> 10 <-> 20

--- 4. Delete head ---
223 <-> 15 <-> 10 <-> 20

--- 5. Delete last ---
223 <-> 15 <-> 10

--- 6. Delete at position 1 ---
223 <-> 10


## Helpers for the problems below
`build_dll` creates a DLL from a Python list, `forward` / `backward` read the list in both directions. Comparing both is a great way to **verify that every `prev` pointer is correct**.

In [3]:
def build_dll(values):
    head = tail = None
    for v in values:
        node = Node(v)
        if head is None:
            head = tail = node
        else:
            tail.next = node
            node.prev = tail
            tail = node
    return head

def forward(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

def backward(head):
    if head is None:
        return []
    while head.next:                 # go to the tail
        head = head.next
    out = []
    while head:
        out.append(head.val)
        head = head.prev
    return out

head = build_dll([1, 2, 3])
print(forward(head), backward(head))

[1, 2, 3] [3, 2, 1]


## 2. Reverse a doubly linked list

**Approach 1 – stack (swap values):** push all values on a stack, then walk again overwriting values with `pop()`. O(n) time, O(n) space.

**Approach 2 – swap pointers (optimal):** for each node, **swap its `next` and `prev`**. After the walk, the old tail is the new head. O(n) time, O(1) space.

> 🐞 **Fix:** the earlier code ignored the `head` argument (used `self.head`) and would crash on an empty list. These are now standalone functions that take and return `head`.

In [4]:
def reverse_dll_stack(head):
    stack = []
    temp = head
    while temp:
        stack.append(temp.val)
        temp = temp.next
    temp = head
    while temp:
        temp.val = stack.pop()       # last value comes out first
        temp = temp.next
    return head

def reverse_dll(head):
    curr = head
    new_head = None
    while curr is not None:
        curr.prev, curr.next = curr.next, curr.prev   # swap the two pointers
        new_head = curr                               # last processed node = new head
        curr = curr.prev                              # (old next is now stored in prev)
    return new_head

h = reverse_dll_stack(build_dll([1, 2, 3, 4]))
print("stack  :", forward(h), backward(h))
h = reverse_dll(build_dll([1, 2, 3, 4]))
print("pointer:", forward(h), backward(h))
print("empty  :", reverse_dll(None))

stack  : [4, 3, 2, 1] [1, 2, 3, 4]
pointer: [4, 3, 2, 1] [1, 2, 3, 4]
empty  : None


## 3. Remove all occurrences of a key

Walk the list; whenever `node.val == key`, unlink it:
- if it has a previous node → `prev.next = node.next`
- if it has a next node → `next.prev = node.prev`
- if it was the **head** → move `head` forward

Because each node knows its own `prev`, we don't need to track a separate "previous" variable.

> 🐞 **Fix:** the earlier version set `prev = temp` even after deleting `temp`, so `prev` pointed at a node that was no longer in the list.

In [5]:
def remove_all_occurrences(head, key):
    temp = head
    while temp is not None:
        nxt = temp.next                       # save before we unlink temp
        if temp.val == key:
            if temp.prev:
                temp.prev.next = temp.next
            else:
                head = temp.next              # deleting the head
            if temp.next:
                temp.next.prev = temp.prev
        temp = nxt
    return head

h = remove_all_occurrences(build_dll([10, 4, 10, 5, 10, 10, 6]), 10)
print(forward(h), backward(h))                    # [4, 5, 6] [6, 5, 4]
print(forward(remove_all_occurrences(build_dll([7]), 7)))   # []

[4, 5, 6] [6, 5, 4]
[]


## 4. Find pairs with a given sum

| Approach | Works on | Time | Space |
|---|---|---|---|
| Brute force (two nested walks) | any list | O(n²) | O(1) |
| Hash set (`target − value`) | any list | O(n) | O(n) |
| **Two pointers** (`left` at head, `right` at tail) | **sorted** list | O(n) | **O(1)** |

The two-pointer version is only possible in a DLL because `right` must move **backward** (`right = right.prev`).

Two-pointer logic on a sorted list:
- sum too small → `left = left.next`
- sum too big → `right = right.prev`
- match → record, move both
- stop when the pointers **cross or meet**.

> 🐞 **Fix:** the earlier loop condition `left.val < right.val` skips pairs of equal values like `(2, 2)`. We compare **node positions** instead: stop when `left is right` or `right.next is left`.

In [6]:
def pairs_brute(head, target):
    result = []
    t1 = head
    while t1:
        t2 = t1.next
        while t2:
            if t1.val + t2.val == target:
                result.append([t1.val, t2.val])
            t2 = t2.next
        t1 = t1.next
    return result

def pairs_hash(head, target):
    seen = set()
    result = []
    temp = head
    while temp:
        need = target - temp.val
        if need in seen:
            result.append([need, temp.val])
        seen.add(temp.val)
        temp = temp.next
    return result

def pairs_two_pointers(head, target):          # list must be SORTED
    result = []
    left = right = head
    while right.next:                          # move `right` to the tail
        right = right.next
    while left is not right and right.next is not left:
        total = left.val + right.val
        if total == target:
            result.append([left.val, right.val])
            left = left.next
            right = right.prev
        elif total > target:
            right = right.prev
        else:
            left = left.next
    return result

values = [1, 2, 3, 4, 9]
print("brute :", pairs_brute(build_dll(values), 5))
print("hash  :", pairs_hash(build_dll(values), 5))
print("2-ptr :", pairs_two_pointers(build_dll(values), 5))
print("equal values:", pairs_two_pointers(build_dll([1, 2, 2, 3, 4]), 4))   # [[1, 3], [2, 2]]

brute : [[1, 4], [2, 3]]
hash  : [[2, 3], [1, 4]]
2-ptr : [[1, 4], [2, 3]]
equal values: [[1, 3], [2, 2]]


## 5. Remove duplicates from a **sorted** DLL

Duplicates are adjacent. Compare each node with its next node:
- **equal** → unlink the next node (`curr.next = dup.next`, fix `dup.next.prev = curr`) and **stay** on `curr` (there might be more copies)
- **different** → move forward

**Time:** O(n) · **Space:** O(1)

> 🐞 **Fix:** the earlier version never advanced `curr`, so it looped forever, and it used `prev` pointers incorrectly.

In [7]:
def remove_duplicates_sorted(head):
    curr = head
    while curr is not None and curr.next is not None:
        if curr.val == curr.next.val:
            dup = curr.next
            curr.next = dup.next              # skip the duplicate
            if dup.next:
                dup.next.prev = curr          # keep the prev link consistent
        else:
            curr = curr.next                  # only advance when no duplicate was removed
    return head

h = remove_duplicates_sorted(build_dll([1, 1, 1, 2, 3, 3, 4, 4, 4]))
print(forward(h), backward(h))               # [1, 2, 3, 4] [4, 3, 2, 1]

[1, 2, 3, 4] [4, 3, 2, 1]


---
### ✅ Key takeaways
- In a DLL, **every link change touches two pointers** (`next` and `prev`).
- `prev` lets you delete a node without tracking the previous node and run **two pointers from both ends**.
- Test with `forward()` **and** `backward()`: if both match, your pointers are consistent.
- Save `nxt = temp.next` *before* you unlink `temp`.